# Model 08: ARIMA (Baseline Klasik)


In [ ]:
import numpy as np
import pandas as pd
import time
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load preprocessed data
data = np.load('processed_data.npz')
X_test, y_test = data['X_test'], data['y_test']

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


In [ ]:
print("\n--- Inference: ARIMA ---")
# Karena ARIMA tidak menerima bentuk multi-output 3D seperti Keras, kita menggunakan pendekatan Rolling Forecast.
# Untuk setiap jendela 60 menit (X_test), kita prediksi 1 menit ke depan, sama persis seperti yang model LSTM kerjakan.

pred_media = []
pred_content = []

t_arr = []
start_time = time.time()

# Untuk kecepatan eksperimen, kita jalankan simulasi ini pada seluruh data test (12000+ baris).
# Menggunakan order=(3, 1, 0) yang cukup kuat untuk melibas tren jangka pendek.

for i in range(len(X_test)):
    # Ambil sejarah 60 menit terakhir
    history_media = X_test[i, :, 0]
    history_content = X_test[i, :, 1]
    
    t_s = time.time()
    # Prediksi rps_media dengan AutoReg (ARIMA d=0, q=0) untuk efisiensi komputasi
    model_media = AutoReg(history_media, lags=3)
    res_media = model_media.fit()
    p_media = res_media.predict(start=len(history_media), end=len(history_media))[0]
    
    # Prediksi rps_content
    model_content = AutoReg(history_content, lags=3)
    res_content = model_content.fit()
    p_content = res_content.predict(start=len(history_content), end=len(history_content))[0]
    t_arr.append(time.time() - t_s)
    
    pred_media.append(p_media)
    pred_content.append(p_content)
    
    if i % 1000 == 0:
        print(f"Processed {i}/{len(X_test)} samples...")

print(f"Selesai dalam {time.time() - start_time:.2f} detik")

# Gabungkan hasil prediksi
pred = np.column_stack((pred_media, pred_content))
np.save('pred_ARIMA.npy', pred)

avg_inference_ms = np.mean(t_arr) * 1000



In [ ]:
results = {
    'ARIMA': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)
